In [5]:
"""
================================================================================
Sentinel-2 Image Search (PREVIEW MODE) for Wildfire Monitoring
================================================================================
This script loads a list of wildfire locations (latitude, longitude, date) from a CSV file,
searches the Microsoft Planetary Computer STAC catalog for Sentinel-2 L2A images,
and finds the best "before" and "after" images for each fire based on the following criteria:

- The image must be captured at least 5 days before or after the fire date.
- A 10×10 km square centered on the fire point must lie entirely within the image footprint.
- The cloud cover within that 10×10 km square must be less than 10% (using the Scene Classification Layer).
- Among candidates meeting the criteria, the image closest to the 5‑day minimum is preferred.

The script runs in PREVIEW MODE: it only creates metadata JSON files (no actual image downloads).
To download the actual GeoTIFFs you need to uncomment the appropriate function calls.
When downloading, all bands will be reprojected to the appropriate UTM zone:
  - UTM zone 34N (EPSG:32634) for longitudes 18°E – 24°E (western Bulgaria)
  - UTM zone 35N (EPSG:32635) for longitudes 24°E – 30°E (eastern Bulgaria)

The results are saved in:
    D:/data/master_thesis/exports/sentinel2_fire_images/preview

================================================================================
ОПИСАНИЕ НА БЪЛГАРСКИ:
Скриптът зарежда списък с пожари (географска ширина, дължина, дата) от CSV файл,
търси в каталога STAC на Microsoft Planetary Computer спътникови изображения Sentinel-2 L2A
и намира най-добрите "преди" и "след" изображения за всеки пожар по следните критерии:

- Изображението трябва да е заснето поне 5 дни преди или след датата на пожара.
- Квадрат с размери 10×10 км, центриран върху точката на пожара, трябва да попада изцяло в
  границите на изображението.
- Облачността в този квадрат трябва да е под 10% (определя се от слоя SCL).
- Предпочита се изображението, което е най-близко до минималния 5‑дневен интервал.

Скриптът работи в режим ПРЕГЛЕД (PREVIEW): създават се само метаданни във формат JSON
(без реално изтегляне на изображения). За да изтеглите GeoTIFF файловете,
трябва да разкоментирате извикванията на съответните функции.
При изтегляне всички канали ще бъдат препроектирани в подходящата UTM зона:
  - UTM зона 34N (EPSG:32634) за дължини 18°E – 24°E (западна България)
  - UTM зона 35N (EPSG:32635) за дължини 24°E – 30°E (източна България)

Резултатите се записват в:
    D:/data/master_thesis/exports/sentinel2_fire_images/preview
================================================================================
"""

import pandas as pd
import pystac_client
import planetary_computer
from datetime import datetime, timedelta
import os
import json
import warnings
import math
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
from pyproj import Transformer
from rasterio.windows import Window

# Suppress noisy deprecation warning
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"get_items\(\) is deprecated"
)

# =============================================================================
# CONFIGURATION – Update the paths as needed
# =============================================================================
INPUT_CSV = r"D:\data\master_thesis\input\fires_suggestion.csv"
OUTPUT_DIR = r"D:\data\master_thesis\exports\sentinel2_fire_images\preview"
SQUARE_SIZE_KM = 10          # side length of the square area to check
MAX_SEARCH_DAYS = 30         # how far beyond the 5‑day minimum to search

# UTM zones for Bulgaria (used when downloading images)
# Western Bulgaria (18°E – 24°E) -> EPSG:32634
# Eastern Bulgaria (24°E – 30°E) -> EPSG:32635
UTM_ZONE_WEST = 32634
UTM_ZONE_EAST = 32635

# Make sure the output folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Sentinel-2 band information (for reference, not all used in preview)
# =============================================================================
SENTINEL2_BANDS = {
    # 10m resolution bands
    "B02": {"name": "Blue", "resolution": 10, "wavelength": "490nm"},
    "B03": {"name": "Green", "resolution": 10, "wavelength": "560nm"},
    "B04": {"name": "Red", "resolution": 10, "wavelength": "665nm"},
    "B08": {"name": "NIR", "resolution": 10, "wavelength": "842nm"},
    # 20m resolution bands
    "B05": {"name": "Red Edge 1", "resolution": 20, "wavelength": "705nm"},
    "B06": {"name": "Red Edge 2", "resolution": 20, "wavelength": "740nm"},
    "B07": {"name": "Red Edge 3", "resolution": 20, "wavelength": "783nm"},
    "B8A": {"name": "Red Edge 4", "resolution": 20, "wavelength": "865nm"},
    "B11": {"name": "SWIR 1", "resolution": 20, "wavelength": "1610nm"},
    "B12": {"name": "SWIR 2", "resolution": 20, "wavelength": "2190nm"},
    # 60m resolution bands
    "B01": {"name": "Coastal Aerosol", "resolution": 60, "wavelength": "443nm"},
    "B09": {"name": "Water Vapor", "resolution": 60, "wavelength": "945nm"},
    "B10": {"name": "Cirrus", "resolution": 60, "wavelength": "1375nm"},
    # Classification bands
    "SCL": {"name": "Scene Classification", "resolution": 20, "wavelength": "N/A"},
    "AOT": {"name": "Aerosol Optical Thickness", "resolution": 20, "wavelength": "N/A"},
    "WVP": {"name": "Water Vapor", "resolution": 20, "wavelength": "N/A"},
}

# Priority bands for land cover classification (order of importance)
PRIORITY_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12", "B05", "B06", "B07", "B8A", "SCL"]


# =============================================================================
# Helper functions
# =============================================================================
def get_utm_epsg(lon: float) -> int:
    """Return the appropriate UTM zone EPSG code for a given longitude."""
    if 18 <= lon < 24:
        return UTM_ZONE_WEST   # EPSG:32634
    elif 24 <= lon <= 30:
        return UTM_ZONE_EAST   # EPSG:32635
    else:
        # For points outside Bulgaria, fall back to generic zone calculation
        zone = math.floor((lon + 180) / 6) + 1
        return 32600 + zone if zone > 0 else 32700 - zone  # 326xx for north, 327xx for south


def create_square_bbox(lat: float, lon: float, size_km: float = SQUARE_SIZE_KM):
    """Return (min_lon, min_lat, max_lon, max_lat) for a square area with given side length."""
    R = 6371.0  # Earth radius (km)

    lat_offset = (size_km / 2) / R * (180 / math.pi)
    min_lat = lat - lat_offset
    max_lat = lat + lat_offset

    lon_offset = (size_km / 2) / (R * math.cos(math.radians(lat))) * (180 / math.pi)
    min_lon = lon - lon_offset
    max_lon = lon + lon_offset

    return (min_lon, min_lat, max_lon, max_lat)


def point_in_granule(item, lat: float, lon: float):
    """Check if the fire point is within the granule's footprint."""
    try:
        from shapely.geometry import Point, shape
        geometry = item.geometry
        if geometry and geometry['type'] == 'Polygon':
            point = Point(lon, lat)
            granule_poly = shape(geometry)
            return granule_poly.contains(point)
        return True
    except ImportError:
        print("  Warning: shapely not available, skipping geometry check")
        return True
    except Exception as e:
        print(f"  Warning: geometry check failed: {e}")
        return True


def get_available_bands(item):
    """Get list of available bands in the STAC item, sorted by priority."""
    available_bands = []
    for band_key in item.assets:
        if any(standard_band in band_key for standard_band in SENTINEL2_BANDS.keys()):
            available_bands.append(band_key)

    # Sort by priority
    available_bands_sorted = []
    for priority_band in PRIORITY_BANDS:
        if priority_band in available_bands:
            available_bands_sorted.append(priority_band)
    for band in available_bands:
        if band not in available_bands_sorted:
            available_bands_sorted.append(band)

    return available_bands_sorted


def calculate_square_cloud_cover(item, center_lat, center_lon):
    """Calculate cloud cover percentage inside the 10×10 km square using the SCL band."""
    try:
        bbox = create_square_bbox(center_lat, center_lon)
        min_lon, min_lat, max_lon, max_lat = bbox

        scl_asset_key = "SCL"
        if scl_asset_key not in item.assets:
            print("    → No SCL band found, using granule cloud cover")
            return item.properties.get("eo:cloud_cover", 100)

        signed_url = planetary_computer.sign(item.assets[scl_asset_key].href)

        with rasterio.open(signed_url) as src:
            transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            min_x, min_y = transformer.transform(min_lon, min_lat)
            max_x, max_y = transformer.transform(max_lon, max_lat)

            row_min, col_min = src.index(min_x, max_y)
            row_max, col_max = src.index(max_x, min_y)

            height, width = src.shape
            row_min = max(0, row_min)
            row_max = min(height - 1, row_max)
            col_min = max(0, col_min)
            col_max = min(width - 1, col_max)

            if (row_max - row_min <= 0) or (col_max - col_min <= 0):
                print("    → Square not entirely within image bounds")
                return 100

            window = Window(col_min, row_min, col_max - col_min, row_max - row_min)
            scl_data = src.read(1, window=window)

            # SCL classes: 3=cloud shadow, 8=medium cloud, 9=high cloud, 10=thin cirrus
            cloud_classes = [3, 8, 9, 10]
            cloud_pixels = np.sum(np.isin(scl_data, cloud_classes))
            total_pixels = scl_data.size

            cloud_percentage = (cloud_pixels / total_pixels) * 100 if total_pixels > 0 else 100
            print(f"    → Square cloud cover: {cloud_percentage:.1f}%")
            return cloud_percentage

    except Exception as e:
        print(f"    → Error calculating square cloud cover: {e}")
        return item.properties.get("eo:cloud_cover", 100)


def find_best_image(catalog, bbox, fire_date, center_lat, center_lon, search_type="before"):
    """
    Find the best image that meets all criteria (≥5 days away, square inside, <10% clouds).
    Returns (item, days_diff) or (None, inf).
    """
    best_item = None
    best_days_diff = float('inf')

    for days_from_minimum in range(0, MAX_SEARCH_DAYS + 1):
        if search_type == "before":
            days_diff = 5 + days_from_minimum
            target_date = fire_date - timedelta(days=days_diff)
        else:
            days_diff = 5 + days_from_minimum
            target_date = fire_date + timedelta(days=days_diff)

        if days_diff > best_days_diff:
            break

        print(f"  Checking {search_type} {days_diff} day(s): {target_date.date()}")

        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox,
                datetime=f"{target_date.strftime('%Y-%m-%d')}/{target_date.strftime('%Y-%m-%d')}",
                query={"eo:cloud_cover": {"lt": 50}},
            )
            items = list(search.items())

            if items:
                valid_items = [item for item in items if point_in_granule(item, center_lat, center_lon)]
                if valid_items:
                    sorted_items = sorted(valid_items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
                    for item in sorted_items[:3]:
                        square_cloud = calculate_square_cloud_cover(item, center_lat, center_lon)
                        if square_cloud < 10:
                            best_item = item
                            best_days_diff = days_diff
                            print(f"    ✓ Found perfect match ({days_diff} days, {square_cloud:.1f}% clouds)")
                            return best_item, best_days_diff
                        else:
                            print(f"    → Square cloud cover {square_cloud:.1f}% > 10%, skipping")
        except Exception as e:
            print(f"    → Search error: {e}")

    # Fallback: no <10% cloud image, find the least cloudy one
    if best_item is None:
        print(f"  No images with <10% square cloud cover found, searching for best available...")
        for days_from_minimum in range(0, MAX_SEARCH_DAYS + 1):
            if search_type == "before":
                days_diff = 5 + days_from_minimum
                target_date = fire_date - timedelta(days=days_diff)
            else:
                days_diff = 5 + days_from_minimum
                target_date = fire_date + timedelta(days=days_diff)

            try:
                search = catalog.search(
                    collections=["sentinel-2-l2a"],
                    bbox=bbox,
                    datetime=f"{target_date.strftime('%Y-%m-%d')}/{target_date.strftime('%Y-%m-%d')}",
                    query={"eo:cloud_cover": {"lt": 100}},
                )
                items = list(search.items())
                if items:
                    valid_items = [item for item in items if point_in_granule(item, center_lat, center_lon)]
                    if valid_items:
                        best_candidate = None
                        lowest_cloud = 100
                        for item in valid_items[:5]:
                            square_cloud = calculate_square_cloud_cover(item, center_lat, center_lon)
                            if square_cloud < lowest_cloud:
                                lowest_cloud = square_cloud
                                best_candidate = item
                                best_days_diff = days_diff
                        if best_candidate:
                            best_item = best_candidate
                            print(f"    → Best available: {days_diff} days, {lowest_cloud:.1f}% clouds")
                            return best_item, best_days_diff
            except Exception as e:
                print(f"    → Search error: {e}")

    return best_item, best_days_diff


def create_preview_metadata(item, output_path, center_lat, center_lon, days_diff, search_type):
    """Create a JSON file with metadata about the suitable image (no actual download)."""
    try:
        available_bands = get_available_bands(item)
        square_cloud = calculate_square_cloud_cover(item, center_lat, center_lon)

        # Determine the recommended UTM EPSG for this point (for later download)
        recommended_epsg = get_utm_epsg(center_lon)

        metadata = {
            "preview_mode": True,
            "filename_preview": f"{search_type}_{output_path.split('_')[1]}_{item.datetime.strftime('%Y%m%d') if item.datetime else 'unknown'}_all_bands.tif",
            "item_id": item.id,
            "acquisition_date": item.datetime.isoformat() if item.datetime else None,
            "days_difference": days_diff,
            "image_type": search_type,
            "cloud_cover": {
                "granule": item.properties.get("eo:cloud_cover", "N/A"),
                "square_20km": square_cloud,
            },
            "center_coordinates": {
                "latitude": center_lat,
                "longitude": center_lon,
            },
            "search_area_km": SQUARE_SIZE_KM,
            "properties": {
                "platform": item.properties.get("platform", "N/A"),
                "constellation": item.properties.get("constellation", "N/A"),
                "processing_level": item.properties.get("processing:level", "N/A"),
            },
            "bands_available": available_bands,
            "band_count": len(available_bands),
            "download_info": {
                "status": "PREVIEW_ONLY",
                "message": "Image meets criteria – download commented out for preview",
                "stac_url": f"https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/{item.id}",
                "file_would_be_saved_as": f"{search_type}_{output_path.split('_')[1]}_{item.datetime.strftime('%Y%m%d') if item.datetime else 'unknown'}_all_bands.tif",
                "recommended_utm_epsg": recommended_epsg,
                "note": "When downloading, reproject all bands to this UTM EPSG for consistent grid",
            },
            "band_details": {},
        }

        for band_key in available_bands:
            band_info = SENTINEL2_BANDS.get(band_key, {"name": band_key, "resolution": "unknown", "wavelength": "unknown"})
            metadata["band_details"][band_key] = band_info

        with open(output_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"  → PREVIEW: Would download {len(available_bands)} bands as: {metadata['filename_preview']}")
        print(f"  → Square cloud cover: {square_cloud:.1f}%")
        print(f"  → STAC Item: {item.id}")
        print(f"  → Recommended UTM EPSG for download: {recommended_epsg}")

        return metadata

    except Exception as e:
        print(f"  → Error creating preview metadata: {e}")
        return None


# =============================================================================
# 1. Load and clean the wildfire list
# =============================================================================
print("Loading wildfire data...")
file_path = INPUT_CSV

if not os.path.exists(file_path):
    raise FileNotFoundError(f"CSV file not found: {file_path}")

df = pd.read_csv(file_path)

# Print column names to help verify the correct ones
print("Columns in the file:")
print(df.columns.tolist())

# UPDATE THESE WITH THE EXACT COLUMN NAMES FROM THE OUTPUT ABOVE
lat_col = 'Lat'      # e.g., 'latitude', 'LAT', 'Latitude'
lon_col = 'Lon'      # e.g., 'longitude', 'LON', 'Longitude'
date_col = 'Date'    # e.g., 'date', 'FIRE_DATE', 'Date '

# Extract only the three columns
extracted_df = df[[lat_col, lon_col, date_col]].copy()

# Drop rows with missing values
extracted_df = extracted_df.dropna()

# Convert Date column to datetime
extracted_df[date_col] = pd.to_datetime(extracted_df[date_col], errors='coerce')

print(f"Done! Loaded and cleaned {len(extracted_df)} rows into 'extracted_df'.\n")


# =============================================================================
# 2. Search for suitable Sentinel-2 images
# =============================================================================
print("PREVIEW MODE: Searching for suitable Sentinel-2 images")
print("DOWNLOAD FUNCTIONS ARE COMMENTED OUT - PREVIEW ONLY")
print("Criteria:")
print("  - Before image: ≥5 days before fire (closer to 5 preferred)")
print("  - After image: ≥5 days after fire (closer to 5 preferred)")
print("  - Square (10km) entirely within image")
print("  - Cloud cover < 10% in square area")
print("  - When downloading: reproject to UTM zone 34N (west) or 35N (east)")
print("=" * 70)

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

successes = []

for idx, row in extracted_df.iterrows():
    lat = row[lat_col]
    lon = row[lon_col]
    date = row[date_col]

    print(f"\n{'='*50}")
    print(f"Fire {idx+1}/{len(extracted_df)} | ({lat:.5f}, {lon:.5f}) | {date.date()}")

    bbox = create_square_bbox(lat, lon)
    print(f"  Search area: {SQUARE_SIZE_KM}km square centered on fire")
    utm_epsg = get_utm_epsg(lon)
    print(f"  This point falls into UTM EPSG:{utm_epsg}")

    # Before image
    print("  Searching for BEFORE image (≥5 days before)...")
    before_item, before_days_diff = find_best_image(catalog, bbox, date, lat, lon, search_type="before")

    if before_item:
        before_date = before_item.datetime.strftime("%Y%m%d")
        metadata_path = os.path.join(OUTPUT_DIR, f"preview_before_{idx+1:03d}_{before_date}.json")
        metadata = create_preview_metadata(before_item, metadata_path, lat, lon, before_days_diff, "before")

        if metadata:
            successes.append({
                "type": "before",
                "fire_id": idx,
                "preview_path": metadata_path,
                "item_id": before_item.id,
                "days_difference": before_days_diff,
                "acquisition_date": before_item.datetime.isoformat() if before_item.datetime else None,
                "granule_cloud_cover": before_item.properties.get("eo:cloud_cover", "N/A"),
                "square_cloud_cover": metadata["cloud_cover"]["square_20km"],
                "bands_available": metadata["bands_available"],
                "band_count": len(metadata["bands_available"]),
                "download_status": "PREVIEW_ONLY",
                "would_be_saved_as": metadata["filename_preview"],
                "utm_epsg": utm_epsg,
            })
            print(f"  ✓ PREVIEW: Suitable before image found {before_days_diff} days before fire")
    else:
        print("  ✗ No suitable before image found")

    # After image
    print("  Searching for AFTER image (≥5 days after)...")
    after_item, after_days_diff = find_best_image(catalog, bbox, date, lat, lon, search_type="after")

    if after_item:
        after_date = after_item.datetime.strftime("%Y%m%d")
        metadata_path = os.path.join(OUTPUT_DIR, f"preview_after_{idx+1:03d}_{after_date}.json")
        metadata = create_preview_metadata(after_item, metadata_path, lat, lon, after_days_diff, "after")

        if metadata:
            successes.append({
                "type": "after",
                "fire_id": idx,
                "preview_path": metadata_path,
                "item_id": after_item.id,
                "days_difference": after_days_diff,
                "acquisition_date": after_item.datetime.isoformat() if after_item.datetime else None,
                "granule_cloud_cover": after_item.properties.get("eo:cloud_cover", "N/A"),
                "square_cloud_cover": metadata["cloud_cover"]["square_20km"],
                "bands_available": metadata["bands_available"],
                "band_count": len(metadata["bands_available"]),
                "download_status": "PREVIEW_ONLY",
                "would_be_saved_as": metadata["filename_preview"],
                "utm_epsg": utm_epsg,
            })
            print(f"  ✓ PREVIEW: Suitable after image found {after_days_diff} days after fire")
    else:
        print("  ✗ No suitable after image found")


# =============================================================================
# 3. Summary and export
# =============================================================================
print(f"\n{'='*70}")
print(f"PREVIEW COMPLETED! Found {len(successes)} suitable images")
print(f"Preview metadata files created in: {OUTPUT_DIR}")
print("NOTE: No actual image files were downloaded - PREVIEW MODE ONLY")
print("To enable downloads, uncomment the save_all_bands_geotiff() function calls")

if successes:
    before_images = [s for s in successes if s["type"] == "before"]
    after_images = [s for s in successes if s["type"] == "after"]

    print(f"\nPREVIEW SUMMARY:")
    print(f"Total suitable images found: {len(successes)}")
    print(f"Preview metadata files created: {len(successes)}")

    all_bands = []
    for success in successes:
        all_bands.extend(success["bands_available"])
    unique_bands = set(all_bands)

    print(f"\nBand availability in suitable images:")
    print(f"  Unique bands available: {len(unique_bands)}")
    print(f"  Bands: {', '.join(sorted(unique_bands))}")

    if before_images:
        avg_before = sum(s["days_difference"] for s in before_images) / len(before_images)
        avg_bands = sum(s["band_count"] for s in before_images) / len(before_images)
        avg_cloud = sum(s["square_cloud_cover"] for s in before_images) / len(before_images)
        print(f"\nBefore images: {len(before_images)}")
        print(f"  Avg days from fire: {avg_before:.1f}")
        print(f"  Avg bands available: {avg_bands:.1f}")
        print(f"  Avg square cloud cover: {avg_cloud:.1f}%")

    if after_images:
        avg_after = sum(s["days_difference"] for s in after_images) / len(after_images)
        avg_bands = sum(s["band_count"] for s in after_images) / len(after_images)
        avg_cloud = sum(s["square_cloud_cover"] for s in after_images) / len(after_images)
        print(f"\nAfter images: {len(after_images)}")
        print(f"  Avg days from fire: {avg_after:.1f}")
        print(f"  Avg bands available: {avg_bands:.1f}")
        print(f"  Avg square cloud cover: {avg_cloud:.1f}%")

    # Save preview results as CSV
    results_df = pd.DataFrame(successes)
    results_csv = os.path.join(OUTPUT_DIR, 'preview_search_results.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\nPreview results saved to: {results_csv}")

    # List created preview JSON files
    print(f"\nCreated preview files:")
    preview_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('preview_') and f.endswith('.json')]
    for file in sorted(preview_files):
        print(f"  📄 {file}")

    print(f"\nTo download these images, uncomment the save_all_bands_geotiff() function calls")
    print(f"in the script and run again.")
    print(f"When downloading, remember to reproject each band to the recommended UTM EPSG")
    print(f"(either 32634 for western Bulgaria or 32635 for eastern Bulgaria).")

Loading wildfire data...
Columns in the file:
['Fire_id', 'Region', 'Population_center', 'Year', 'Lon', 'Lat', 'Date']
Done! Loaded and cleaned 16 rows into 'extracted_df'.

PREVIEW MODE: Searching for suitable Sentinel-2 images
DOWNLOAD FUNCTIONS ARE COMMENTED OUT - PREVIEW ONLY
Criteria:
  - Before image: ≥5 days before fire (closer to 5 preferred)
  - After image: ≥5 days after fire (closer to 5 preferred)
  - Square (10km) entirely within image
  - Cloud cover < 10% in square area
  - When downloading: reproject to UTM zone 34N (west) or 35N (east)

Fire 1/16 | (41.91079, 26.06846) | 2025-08-04
  Search area: 10km square centered on fire
  This point falls into UTM EPSG:32635
  Searching for BEFORE image (≥5 days before)...
  Checking before 5 day(s): 2025-07-30
  Checking before 6 day(s): 2025-07-29
  Checking before 7 day(s): 2025-07-28
    → Square cloud cover: 0.0%
    ✓ Found perfect match (7 days, 0.0% clouds)
    → Square cloud cover: 0.0%
  → PREVIEW: Would download 15 band